### Descobrir maior fatia


In [ ]:
import os
from tqdm import tqdm
import nibabel as nib
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1 import ImageGrid

def load_nifti_data_balanced(base_dir, class_names, target=1000):
    images = []
    labels = []
    paths = []
    
    # Caminhos das subpastas
    for label in class_names:
        print(f"carregando diretório {label}")
        label_dir = os.path.join(base_dir, label)
        count = 0

        names = os.listdir(label_dir)
        for fname in names:
            img_path = os.path.join(label_dir, fname)
            img = nib.load(img_path).get_fdata()
            images.append(img)
            labels.append(label)
            paths.append(img_path)
            count += 1

        print(f"diretório carregado {count}")
    
    return images, labels, paths

def load_nifti_paths(base_dir, class_names, target=1000):
    paths = []
    
    # Caminhos das subpastas
    for label in class_names:
        print(f"carregando diretório {label}")
        label_dir = os.path.join(base_dir, label)
        count = 0
        names = os.listdir(label_dir)

        for fname in names:
            img_path = os.path.join(label_dir, fname)
            paths.append(img_path)
            count += 1

        print(f"diretório carregado {count}")
    
    return paths

def encontrar_maior_fatia_single(img, eixo=2):
    non_black_pixels, idx_biggest, atual = 0, 0, 0

    for j in range(0, img.shape[eixo]):
        atual = np.count_nonzero(img[:, :, j])

        if (atual > non_black_pixels):
            non_black_pixels = atual
            idx_biggest = j

    return idx_biggest

def encontrar_maior_fatia_dir(testing_imgs, eixo=2):
    biggest_slices = [0 for z in range(0, testing_imgs[0].shape[eixo])]

    for bound_image in testing_imgs:
        idx_biggest = encontrar_maior_fatia_single(bound_image, 2)                
        biggest_slices[idx_biggest] += 1

    return biggest_slices

In [ ]:
adni_path_pre = "/mnt/c/Users/Paulo Pires/Desktop/Alzheimer_cnn/ADNI/NIFTI_PROCESSED/test"
adni_path_post = "/mnt/c/Users/Paulo Pires/Desktop/Alzheimer_cnn/ADNI/ADNI_NORMALIZED/test"
# testing_path = "/mnt/c/Users/Paulo Pires/Desktop/Alzheimer_cnn/OASIS/OASIS_PRE_NORMALIZED_MR2"

oasis_path_post_val = "/mnt/c/Users/Paulo Pires/Desktop/Alzheimer_cnn/OASIS/OASIS_NORMALIZED_MR2/validation"
oasis_path_post_train = "/mnt/c/Users/Paulo Pires/Desktop/Alzheimer_cnn/OASIS/OASIS_NORMALIZED_MR2/train"
oasis_path_pre = "/mnt/c/Users/Paulo Pires/Desktop/Alzheimer_cnn/OASIS/OASIS_PRE_NORMALIZED_MR2"

labels_adni = ['cn', 'emci', 'mci', 'lmci', 'ad']
labels_oasis = ['0.0', '0.5', '1.0', '2.0']

# testing_imgs, testing_labels, testing_names = load_nifti_data_balanced(testing_path, labels)
# adni_names_pre = load_nifti_paths(adni_path_pre, labels_adni)
# adni_names_post = load_nifti_paths(adni_path_post, labels_adni)

# oasis_names_pre = load_nifti_paths(oasis_path_pre, labels_oasis)
oasis_names_post = load_nifti_paths(oasis_path_post_train, labels_oasis) + load_nifti_paths(oasis_path_post_val, labels_oasis)
# oasis_names_post = sorted(oasis_names_post)

fsl_dir_train = "/mnt/c/Users/Paulo Pires/Desktop/Alzheimer_cnn/OASIS/OASIS_1_FSL_SEP/train"
fsl_dir_val = "/mnt/c/Users/Paulo Pires/Desktop/Alzheimer_cnn/OASIS/OASIS_1_FSL_SEP/validation"

fsl_names_train = load_nifti_paths(fsl_dir_train, labels_oasis)
fsl_names_validation = load_nifti_paths(fsl_dir_val, labels_oasis)

# biggest_slices = encontrar_maior_fatia_dir(testing_imgs, 2)
# adni-> 77

# bigbig = max(biggest_slices)
# big_index = biggest_slices.index(bigbig)
# print(f"maior fatia: {big_index} com {bigbig}")
# print(biggest_slices)
# print(len(adni_names_pre))
# print(len(oasis_names_post))
# print(len(oasis_names_pre))

In [ ]:
print(nib.load(oasis_names_post[0]).get_fdata().shape)

In [ ]:
# Normalization -> valores de voxels entre 0 e 1
def normalize_image_min(image_data): 
    min_val = np.min(image_data)
    max_val = np.max(image_data)
    normalized_data = (image_data - min_val) / (max_val - min_val)
    return normalized_data

def winsorize_image(image_data, lower_percentile=0, upper_percentile=99.9):
    lower_bound = np.percentile(image_data, lower_percentile)
    upper_bound = np.percentile(image_data, upper_percentile)
    winsorized_data = np.clip(image_data, lower_bound, upper_bound)
    return winsorized_data

In [ ]:
def metricas_imagem(data, title): #print metricas de uma imagem (max, min, media)
    if len(data.shape) > 1:
        values = data.flatten()
    else:
        values = data

    # print(f"MEDIA: {np.mean(values)}")
    # print(f"DESVIO: {np.std(values)}")
    # print(f"MIN: {np.min(values)}")
    # print(f"MAX: {np.max(values)}")

    plt.hist(values)
    plt.title(f"{title} | Média: {np.mean(values)}")
    plt.show()

def unificar_tamanhos_com_padding(lista_de_imagens):
    max_altura = 0
    max_largura = 0
    for img in lista_de_imagens:
        altura, largura = img.shape
        if altura > max_altura:
            max_altura = altura
        if largura > max_largura:
            max_largura = largura

    imagens_uniformes = []
    for img in lista_de_imagens:
        fundo = np.zeros((max_altura, max_largura))
        
        altura_img, largura_img = img.shape
        y_offset = (max_altura - altura_img) // 2
        x_offset = (max_largura - largura_img) // 2
        
        fundo[y_offset:y_offset+altura_img, x_offset:x_offset+largura_img] = img
        imagens_uniformes.append(fundo)
        
    return imagens_uniformes

def plot_views_uniforme_final(image, main_title, k=0, sag_idx=90, cor_idx=110, ax_idx=110, figsize=(15, 5), axes_pad=0.3):
    fig = plt.figure(figsize=figsize)
    grid = ImageGrid(fig, 111,
                    nrows_ncols=(1, 3),
                    axes_pad=axes_pad)
    
    slices_originais = [
        np.rot90(image[sag_idx, :, :], k=k),
        np.rot90(image[:, cor_idx, :], k=k),
        np.rot90(image[:, :, ax_idx], k=k)
    ]
    
    slices_uniformizadas = unificar_tamanhos_com_padding(slices_originais)
    
    titles = ["Sagital", "Coronal", "Axial"]

    for ax, im_slice, title in zip(grid, slices_uniformizadas, titles):
        ax.imshow(im_slice, cmap='gray')
        ax.set_title(title)
        ax.axis('off')

    fig.suptitle(main_title)
    plt.show()

In [ ]:
adni_pre = nib.load(fsl_names_train[8]).get_fdata()[:, :, :, 0]
# adni_post = nib.load(adni_names_post[8]).get_fdata()

metricas_imagem(adni_pre, "adni_pre_normalização e wins")
# metricas_imagem(adni_post, "adni_pos_normalização e wins")

plot_views_uniforme_final(adni_pre, "adni_pre_normalização e wins")
# plot_views_uniforme_final(adni_post, "adni_pos_normalização e wins")

oasis_pre = nib.load(oasis_names_pre[8]).get_fdata()
oasis_post = nib.load(oasis_names_post[8]).get_fdata()

metricas_imagem(oasis_pre, "oasis_pre_normalização e wins")
metricas_imagem(oasis_post, "oasis_pos_normalização e wins")

plot_views_uniforme_final(oasis_pre, "oasis_pre_normalização e wins")
plot_views_uniforme_final(oasis_post, "oasis_pos_normalização e wins")

In [ ]:
oasis_pre = nib.load(oasis_names_pre[8]).get_fdata()
oasis_post = nib.load(oasis_names_post[8]).get_fdata()

metricas_imagem(oasis_pre, "oasis_pre_normalização e wins")
metricas_imagem(oasis_post, "oasis_pos_normalização e wins")

plot_views_uniforme_final(oasis_pre, "oasis_pre_normalização e wins")
plot_views_uniforme_final(oasis_post, "oasis_pos_normalização e wins")

In [ ]:
wins_data = winsorize_image(testing_pre, 0, 99.9)
plot_views_uniforme_final(testing_pre, "adni_pre_normalização e wins")
plot_views_uniforme_final(wins_data, "adni_pos_normalização e wins")

### Encontrar índices para cortar

In [ ]:
import os
from tqdm import tqdm
import nibabel as nib
import numpy as np
import matplotlib.pyplot as plt

def encontrar_bordas_xy(data):
    min_x, max_x, min_y, max_y, min_z, max_z = 0, data.shape[0]-1, 0, data.shape[1]-1, 0, data.shape[2]-1
    while (np.count_nonzero(data[min_x, :, :])) == 0:
        min_x += 1
    min_x -= 5
    #print(min_x)

    while (np.count_nonzero(data[max_x, :, :])) == 0:
        max_x -= 1
    max_x += 5
    #print(max_x)

    while (np.count_nonzero(data[:, min_y, :])) == 0:
        min_y += 1
    min_y -= 5
    #print(min_y)

    while (np.count_nonzero(data[:, max_y, :])) == 0:
        max_y -= 1
    max_y += 5
    #print(max_y)

    while (np.count_nonzero(data[:, :, min_z])) == 0:
        min_z += 1
    min_z -= 5
    #print(max_y)

    while (np.count_nonzero(data[:, :, max_z])) == 0:
        max_z -= 1
    max_z += 5
    #print(max_y)

    return min_x, max_x, min_y, max_y, min_z, max_z

test_sample = testing_imgs[0]

plt.imshow(testing_imgs[0][:, :, big_index], cmap='gray')
plt.show()

x1, x2, y1, y2, z1, z2 = encontrar_bordas_xy(testing_imgs[0])
print(x1,x2, y1, y2)

plt.imshow(testing_imgs[0][x1:x2, y1:y2, big_index], cmap='gray')
plt.show()

In [ ]:
for cropping_image in testing_imgs:
    minx1, minx2, miny1, miny2, minz1, minz2 = 100, 100, 100, 100, 100, 100
    x1, x2, y1, y2, z1, z2 = encontrar_bordas_xy(cropping_image)

    minx1 = min(minx1, x1)
    minx2 = max(minx2, x2)
    miny1 = min(miny1, y1)
    miny2 = max(miny2, y2)
    minz1 = min(minz1, z1)
    minz2 = max(minz2, z2)

#adni -> (160, 189, 157, 1)

#adni novo -> 156, 195, 160

cropped_test_sample = test_sample[x1:x2, y1:y2, z1:z2]
# cropped_test_sample = test_sample[16:176, 18:207, 10:167]

plt.imshow(cropped_test_sample[:, :, big_index], cmap='gray')
plt.title(cropped_test_sample.shape)
plt.show()

plt.imshow(cropped_test_sample[:, 80, :], cmap='gray')
plt.title(cropped_test_sample[:, 80, :].shape)
plt.show()

plt.imshow(cropped_test_sample[80, :, :], cmap='gray')
plt.title(cropped_test_sample[80, :, :].shape)
plt.show()

In [ ]:
img_to_crop = nib.load(fsl_names_train[0]).get_fdata()[:, :, :, 0]
print(img_to_crop.shape)

In [ ]:
x1, x2 = 10, 166
y1, y2 = 10, 205
z1, z2 = 0, 160

# plt.imshow(img_to_crop[x1:x2, y1:y2, 88])
# plt.title(img_to_crop[x1:x2, y1:y2, z1:z2].shape)

plot_views_uniforme_final(img_to_crop, img_to_crop[x1:x2, y1:y2, z1:z2].shape)

#adni novo -> 156, 195, 160
plt.show()

In [ ]:
for cropping_name in tqdm(fsl_names_validation, desc="Processando amostras"):
    cropped_test_sample = nib.load(cropping_name)
    cropping_affine = cropped_test_sample.affine

    nii_img = nib.Nifti1Image(cropped_test_sample.get_fdata()[x1:x2, y1:y2, z1:z2], cropping_affine)

    # Salva como .nii.gz
    nib.save(nii_img, cropping_name)

In [ ]:
img_cropped = nib.load(testing_names[0]).get_fdata()

plt.imshow(img_cropped[:, :, 80])
plt.title(img_cropped.shape)
#adni novo -> 156, 195, 160
plt.show()

In [ ]:
for item in testing_names:
    img_plot = nib.load(item)
    plt.imshow(img_plot.get_data()[:, :, 100])
    plt.title(img_plot.shape)
    plt.show()

In [ ]:

names_to_plot = os.listdir("/mnt/c/Users/Paulo Pires/Desktop/Alzheimer_cnn/NIFTI_PROCESSED/test/ad")
n_item = 0

for item in names_to_plot:
    if n_item < 50:
        img_cropped = nib.load(f"/mnt/c/Users/Paulo Pires/Desktop/Alzheimer_cnn/NIFTI_PROCESSED/test/ad/{item}").get_fdata()
        
        figure, axs = plt.subplots(1, 5)

        axs[0].imshow(img_cropped[:, :, 65], cmap='gray')
        axs[0].axis('off')
        axs[1].imshow(img_cropped[:, :, 75], cmap='gray')
        axs[1].axis('off')
        axs[2].imshow(img_cropped[:, :, 85], cmap='gray')
        axs[2].axis('off')
        axs[3].imshow(img_cropped[:, :, 95], cmap='gray')
        axs[3].axis('off')
        axs[4].imshow(img_cropped[:, :, 105], cmap='gray')
        axs[4].axis('off')

        plt.tight_layout(rect=[0, 0, 1, 1.6])
        plt.suptitle(item)
        plt.show()

names_to_plot = os.listdir('testing')
n_item = 0

for item in names_to_plot:
    if n_item < 100:
        img_cropped = nib.load(f"testing/{item}").get_fdata()
        
        figure, axs = plt.subplots(1, 6)

        axs[0].imshow(img_cropped[:, :, 70], cmap='gray')
        axs[0].axis('off')
        axs[1].imshow(img_cropped[:, :, 75], cmap='gray')
        axs[1].axis('off')
        axs[2].imshow(img_cropped[:, :, 80], cmap='gray')
        axs[2].axis('off')
        axs[3].imshow(img_cropped[:, :, 85], cmap='gray')
        axs[3].axis('off')
        axs[4].imshow(img_cropped[:, :, 90], cmap='gray')
        axs[4].axis('off')
        axs[5].imshow(img_cropped[:, :, 95], cmap='gray')
        axs[5].axis('off')

        plt.tight_layout(rect=[0, 0, 1, 1.7])
        plt.suptitle(item)
        plt.show()
        n_item += 1

subset = ['train', 'validation', 'test']
labels = ['cn', 'emci', 'mci', 'lmci', 'ad']

base_dir = "/mnt/c/Users/Paulo Pires/Desktop/Alzheimer_cnn/NIFTI_PROCESSED"

labels_oasis = ['0.0', '0.5', '1.0']

base_dir_oasis = "/mnt/c/Users/Paulo Pires/Desktop/Alzheimer_cnn/OASIS/OASIS_2_PROCESSED"

for sub in subset:
    dir_sub = f"{base_dir}/{sub}"

    for label in labels:
        dir = f"{dir_sub}/{label}"
        names_to_change = os.listdir(dir)

        print(f"salvando no diretório {sub}/{label}")

        #for item in names_to_change:
        for item in names_to_change:
            file_path = f"{dir}/{item}"
            img_to_crop = nib.load(file_path)
            img_affine = img_to_crop.affine
            img_to_crop = img_to_crop.get_fdata()
            crop_img = img_to_crop[x1:x2, y1:y2, z1:z2]

            nii_img = nib.Nifti1Image(crop_img, img_affine)

            #nib.save(nii_img, file_path)

subset = ['train', 'validation', 'test']
labels = ['cn', 'emci', 'mci', 'lmci', 'ad']

base_dir = "/mnt/c/Users/Paulo Pires/Desktop/Alzheimer_cnn/NIFTI_PROCESSED"

for sub in subset:
    dir_sub = f"{base_dir}/{sub}"

    for label in labels:
        count = 0
        dir = f"{dir_sub}/{label}"
        names_to_plot = os.listdir(dir)

        print(f"salvando no diretório {sub}/{label}")

        for item in names_to_plot:
            if count < 10:
                file_path = f"{dir}/{item}"
                img_plot = nib.load(file_path).get_fdata()
                plt.imshow(img_plot[:, :, 80], cmap='gray')
                plt.title(f"{item} - {sub} - {label}")
                plt.show()
                count += 1

##### CROPPAR IMAGENS (NIFTI)


In [ ]:
import os
import nibabel as nib
import matplotlib.pyplot as plt

# índices de corte para o corte NIfTI
SLICE_NII_IDX0 = slice(24, 169)
SLICE_NII_IDX1 = slice(24, 206)
SLICE_NII_IDX2 = slice(6, 161)

base = 'train'
dir = 'ad'

for subset in os.listdir(base):
    dir = os.path.join(base, subset) # salva junção do conjunto com a pasta referente à classe
    for name in os.listdir(dir):
        file = os.path.join(dir, name) # acessa cada arquivo dentro da pasta
        image = nib.load(file)
        data = image.get_fdata()

        print(f"shape da imagem {name} original: {data.shape}")
        data_cropped = data[SLICE_NII_IDX0, SLICE_NII_IDX1, SLICE_NII_IDX2] # corta a imagem para o tamanho desejadao
        print(f"shape da imagem {name} cortada: {data_cropped.shape}\n\n")

        # criar um novo objeto NIfTI com o corte
        nifti_cropped = nib.Nifti1Image(data_cropped, image.affine, image.header)
        
        # salvar no mesmo arquivo (sobrescrevendo)
        nib.save(nifti_cropped, file)

##### NORMALIZAR DADOS

In [ ]:
import os
import ants
import numpy as np
import matplotlib.pyplot as plt
from concurrent.futures import ThreadPoolExecutor

def normalize_img(input_folder, output_folder, item, lower_percentile=1, upper_percentile=99):
    input_path = os.path.join(input_folder, item)
    output_path = os.path.join(output_folder, item)

    img_3d = ants.image_read(input_path)
    data_3d = img_3d.numpy()

    # Apply Winsorization
    data_3d = winsorize_image(data_3d, lower_percentile, upper_percentile)

    # Normalize the image (without background)
    data_3d = normalize_image_min_except_background(data_3d)

    # Convert the numpy array back to an ANTs image
    img_3d = ants.from_numpy(data_3d)

    # Save the result
    ants.image_write(img_3d, output_path)
    print(f"IMAGEM SALVA EM {output_path}\n")

def normalize_image_min_except_background(data_3d):
    non_zero_voxels = data_3d[data_3d != 0]

    if non_zero_voxels.size > 0:
        min_value = np.min(non_zero_voxels)
        max_value = np.max(data_3d)

        normalized_data = data_3d.copy() 
        
        diff = max_value - min_value

        if diff > 1e-6:

            tissue_mask = data_3d != 0
            
            normalized_data[tissue_mask] = (normalized_data[tissue_mask] - min_value) / diff
            
            normalized_data[data_3d == 0] = 0 
            
        else:
            normalized_data[data_3d != 0] = 0
            
        return normalized_data
    
    return data_3d

def winsorize_image(image_data, lower_percentile=0, upper_percentile=99.9):
    lower_bound = np.percentile(image_data, lower_percentile)
    upper_bound = np.percentile(image_data, upper_percentile)
    winsorized_data = np.clip(image_data, lower_bound, upper_bound)
    return winsorized_data

In [ ]:
# DIR_BASE = "/mnt/c/Users/Team Taiane/Desktop/ADNI/FULL_ADNI/raw_data/3D_BRAIN_NOT_NORMALIZED"
# DIR_NORMALIZED = "/mnt/c/Users/Team Taiane/Desktop/ADNI/FULL_ADNI/3D_NORM"

DIR_BASE = "/mnt/c/Users/Paulo Pires/Desktop/Alzheimer_cnn/ADNI/NIFTI_PROCESSED"
DIR_NORMALIZED = "/mnt/c/Users/Paulo Pires/Desktop/Alzheimer_cnn/ADNI/ADNI_NORMALIZED"


os.makedirs(DIR_NORMALIZED, exist_ok=True)

for subfolder in ['train', 'validation', 'test']:
    if subfolder in os.listdir(DIR_BASE):   
        print(f"\n\nDIRECTORY: {subfolder}\n\n")
        input_dir = os.path.join(DIR_BASE, subfolder)
        output_dir = os.path.join(DIR_NORMALIZED, subfolder)
        os.makedirs(output_dir, exist_ok=True)

        for group in ['cn', 'emci', 'mci', 'lmci', 'ad']:
            if group in os.listdir(input_dir):
                print(f"GROUP: {group}\n\n")
                input_folder = os.path.join(input_dir, group)            
                output_folder = os.path.join(output_dir, group)       
                os.makedirs(output_folder, exist_ok=True)

                already_processed = [file for file in os.listdir(output_folder)] # checa se alguma já foi processada
                image_paths = [file for file in os.listdir(input_folder) if file not in already_processed] # carrega as imagens não processadas

                with ThreadPoolExecutor(max_workers=128) as executor:
                    futures = [executor.submit(normalize_img, input_folder, output_folder, item) for item in image_paths]

                    # Aguarda todas as tarefas terminarem
                    for future in futures:
                        future.result()

In [ ]:
# DIR_BASE = "/mnt/c/Users/Team Taiane/Desktop/ADNI/FULL_ADNI/raw_data/3D_BRAIN_NOT_NORMALIZED"
# DIR_NORMALIZED = "/mnt/c/Users/Team Taiane/Desktop/ADNI/FULL_ADNI/3D_NORM"

DIR_BASE = "/mnt/c/Users/Paulo Pires/Desktop/Alzheimer_cnn/OASIS/OASIS_1_FSL_SEP/train"
DIR_NORMALIZED = "/mnt/c/Users/Paulo Pires/Desktop/Alzheimer_cnn/OASIS_1_FSL_NORMALIZED/train"

os.makedirs(DIR_NORMALIZED, exist_ok=True)

for group in ['0.0', '0.5', '1.0', '2.0']:
    input_dir = DIR_BASE
    output_dir = DIR_NORMALIZED
    if group in os.listdir(input_dir):
        print(f"GROUP: {group}\n\n")
        input_folder = os.path.join(input_dir, group)            
        output_folder = os.path.join(output_dir, group)       
        os.makedirs(output_folder, exist_ok=True)

        already_processed = [file for file in os.listdir(output_folder)] # checa se alguma já foi processada
        image_paths = [file for file in os.listdir(input_folder) if file not in already_processed] # carrega as imagens não processadas

        with ThreadPoolExecutor(max_workers=128) as executor:
            futures = [executor.submit(normalize_img, input_folder, output_folder, item) for item in image_paths]

            # Aguarda todas as tarefas terminarem
            for future in futures:
                future.result()

In [ ]:
DIR_NORMALIZED = "/mnt/c/Users/Paulo Pires/Desktop/Alzheimer_cnn/OASIS_1_FSL_NORMALIZED/validation"

for group in ['2.0']:
    output_dir = DIR_NORMALIZED
    if group in os.listdir(output_dir):
        print(f"GROUP: {group}\n\n")       
        output_folder = os.path.join(output_dir, group)       

        image_paths = [file for file in os.listdir(output_folder)] # carrega as imagens não processadas

        for item in image_paths:
            img = nib.load(os.path.join(output_folder, item)).get_fdata()
            plot_views_uniforme_final(img, os.path.basename(item))
            print(np.max(img))
            print(np.min(img))
            print(np.mean(img))